# DocGuard — Phase 3 CNN baseline

Synthetic-data research prototype. Select **Runtime → Change runtime type → GPU**, then run cells in order. Upload `docguard_colab.zip` when asked. No accuracy is assumed in advance. Preserve downloaded outputs. Training here is user initiated; this notebook has not been run in your Google account.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os
uploaded = files.upload()
assert "docguard_colab.zip" in uploaded, "Upload docguard_colab.zip"
work = Path("/content/docguard")
work.mkdir(exist_ok=True)
with zipfile.ZipFile("docguard_colab.zip") as archive:
    for name in archive.namelist():
        assert (work/name).resolve().is_relative_to(work.resolve()), "Unsafe archive path"
    archive.extractall(work)
os.chdir(work)
print("Project extracted:", work)

In [ ]:
%pip install -q -r requirements-train.txt
import torch
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Unavailable")
assert torch.cuda.is_available(), "Enable a GPU runtime before full training"

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "scripts/validate_dataset.py", "data/synthetic_v2"], check=True)
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)

## Train
Uses all approved training images. The best checkpoint is selected by validation F1 at threshold 0.5. Afterwards the output threshold is selected on validation only. Test data is not read by training. If rerunning, select a new output directory rather than overwriting a previous run.

In [ ]:
import subprocess, sys
RUN = "runs/cnn_colab"
subprocess.run([sys.executable, "-m", "docguard.baseline", "train", "--epochs", "20", "--batch-size", "8", "--output", RUN], check=True)

## Held-out evaluation
Run after training finishes. Do not tune settings using test results. Authentic false alarms use a fixed rule: at least 0.1% of pixels predicted edited. This is not a calibrated document authenticity score.

In [ ]:
subprocess.run([sys.executable, "-m", "docguard.baseline", "test", "--output", RUN], check=True)
import json
print(json.dumps(json.loads(Path(RUN, "test_metrics.json").read_text()), indent=2))

In [ ]:
from IPython.display import display, Image
for path in sorted(Path(RUN, "predictions").glob("*.png"))[:4]:
    display(Image(filename=str(path)))

In [ ]:
import shutil
archive = shutil.make_archive("/content/docguard_cnn_results", "zip", RUN)
files.download(archive)

## Interpretation
Report synthetic-only F1/IoU, authentic false alarms, and per-edit-type metrics. Clean matched-font replacements may contain no recoverable forensic evidence. Generated dates can be invalid and introduce shortcuts. A high synthetic score does not demonstrate real-document accuracy. Keep the dataset and manifest unchanged for the Transformer comparison.